# YouTube Mix to Playlist Converter

This notebook converts YouTube auto-generated mixes into real YouTube playlists.




In [1]:
# STEP 1: INSTALL DEPENDENCIES (Run this cell once)

# These libraries are needed for YouTube extraction and API access
!pip install yt-dlp google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("Dependencies installed!")



Defaulting to user installation because normal site-packages is not writeable
Dependencies installed!


In [2]:
# STEP 2: IMPORT LIBRARIES

import os
import json
import time
from dataclasses import dataclass
from typing import List, Optional

# For extracting YouTube mix data (without downloading videos)
import yt_dlp

# For YouTube Data API v3 authentication and requests
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

print("All libraries imported!")



/Users/jeremyagada/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jeremyagada/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/jeremyagada/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python ve

All libraries imported!


/Users/jeremyagada/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [3]:
# STEP 3: DEFINE DATA STRUCTURES and EXTRACTOR

@dataclass
class Song:
    # Represents a single song extracted from YouTube.
    # Attributes:
    #     title: Song title (e.g., "Bohemian Rhapsody")
    #     artist: Artist name (e.g., "Queen")
    #     youtube_id: YouTube video ID (e.g., "fJ9rUzIMcZQ")
    #     duration: Length in seconds
    title: str
    artist: str
    youtube_id: Optional[str] = None
    duration: Optional[int] = None

    def __str__(self):
        return f"{self.artist} - {self.title}"


class YouTubeExtractor:
    # Extracts song metadata from YouTube mixes using yt-dlp.
    # YouTube mixes are auto-generated playlists (URL contains andlist=RD...).
    # This class reads the mix without downloading any videos.

    def extract_mix(self, mix_url: str, max_songs: int = 50) -> List[Song]:
        # Extract all songs from a YouTube mix URL.
        # Args:
        #     mix_url: YouTube mix URL with andlist=RD... parameter
        #     max_songs: Maximum songs to extract (default 50, YouTube mixes usually have 25-50)
        # Returns:
        #     List of Song objects with artist, title, youtube_id, duration

        # yt-dlp options:
        #   quiet=True          -> Suppress console output
        #   extract_flat=True   -> Get metadata only, no download
        #   playlistend=50      -> Limit to first N songs
        #   ignoreerrors=True   -> Skip unavailable/private videos
        ydl_opts = {
            'quiet': True,
            'extract_flat': True,
            'playlistend': max_songs,
            'ignoreerrors': True,
        }

        songs = []

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Extracting from: {mix_url[:60]}...")

            # This fetches the playlist metadata WITHOUT downloading anything
            info = ydl.extract_info(mix_url, download=False)

            # 'entries' contains each video in the mix
            if 'entries' not in info:
                print("No playlist entries found. Make sure the URL has andlist=RD...")
                return songs

            for entry in info['entries']:
                if not entry:
                    continue  # Skip deleted/private videos

                raw_title = entry.get('title', 'Unknown')
                video_id = entry.get('id')
                duration = entry.get('duration')

                # YouTube titles often follow "Artist - Title" format
                # We try to split them for cleaner playlist data
                artist, song_title = self._parse_title(raw_title)

                songs.append(Song(
                    title=song_title,
                    artist=artist,
                    youtube_id=video_id,
                    duration=duration
                ))

        print(f"Extracted {len(songs)} songs from the mix!")
        return songs

    def _parse_title(self, title: str) -> tuple:
        # Parse YouTube video titles into (artist, title) pairs.
        # Handles formats like:
        #   "Queen - Bohemian Rhapsody"
        #   "Queen Bohemian Rhapsody" (en-dash)
        #   "Queen Bohemian Rhapsody" (em-dash)
        #   "Queen | Bohemian Rhapsody"
        # Returns:
        #     (artist, title) tuple. If unparseable, returns ("Unknown", title)

        separators = [' - ', ' – ', ' — ', ' | ']

        for sep in separators:
            if sep in title:
                parts = title.split(sep, 1)
                if len(parts) == 2:
                    return parts[0].strip(), parts[1].strip()

        # Fallback: could not parse artist from title
        return "Unknown", title


# Test the extractor (we will use it in the next cell)
print("YouTubeExtractor class defined!")
print("   Usage: extractor = YouTubeExtractor()")
print("          songs = extractor.extract_mix('YOUR_MIX_URL')")



YouTubeExtractor class defined!
   Usage: extractor = YouTubeExtractor()
          songs = extractor.extract_mix('YOUR_MIX_URL')


In [ ]:
# STEP 4: YOUTUBE PLAYLIST CREATOR (API)

class YouTubePlaylistCreator:
    # Creates YouTube playlists and adds songs using YouTube Data API v3.
    # Requires:
    #     - client_secret.json (OAuth 2.0 credentials from Google Cloud)
    #     - YouTube Data API v3 enabled in your Google Cloud project

    # OAuth scope: allows creating playlists and adding videos
    SCOPES = ['https://www.googleapis.com/auth/youtube']

    def __init__(self, client_secrets_file: str = "client_secret.json"):
        # Initialize and authenticate with YouTube API.
        # This will open a browser tab asking you to grant permission.
        # You will only need to do this once per session.
        # Args:
        #     client_secrets_file: Path to your downloaded client_secret.json
        self.youtube = self._authenticate(client_secrets_file)
        print("Successfully authenticated with YouTube API!")

    def _authenticate(self, secrets_file: str):
        # OAuth 2.0 authentication flow.
        # Steps:
        # 1. Reads client_secret.json
        # 2. Opens browser for user consent
        # 3. Receives auth token via local callback server
        # 4. Returns authenticated YouTube API service
        flow = InstalledAppFlow.from_client_secrets_file(
            secrets_file, 
            self.SCOPES
        )
        # port=0 lets the OS pick an available port
        credentials = flow.run_local_server(port=0)

        # 'youtube' is the service object for all API calls
        return build('youtube', 'v3', credentials=credentials)

    def create_playlist(self, title: str, description: str = "", privacy: str = "private") -> str:
        # Create a new YouTube playlist in your account.
        # Args:
        #     title: Playlist name (e.g., "My Awesome Mix")
        #     description: Optional description shown on YouTube
        #     privacy: One of 'private' (only you), 'unlisted' (link only), 'public' (everyone)
        # Returns:
        #     Playlist ID string (used to add songs later)
        request = self.youtube.playlists().insert(
            part="snippet,status",
            body={
                "snippet": {
                    "title": title,
                    "description": description
                },
                "status": {
                    "privacyStatus": privacy
                }
            }
        )
        response = request.execute()
        playlist_id = response['id']

        print(f"Created YouTube playlist: '{title}'")
        print(f"   Privacy: {privacy}")
        print(f"   ID: {playlist_id}")
        return playlist_id

    def add_songs(self, playlist_id: str, songs: List[Song]):


        print(f"Adding {len(songs)} songs to playlist...")

        for i, song in enumerate(songs, 1):
            # Search query combines artist and title
            search_query = f"{song.artist} {song.title}"

            search_response = self.youtube.search().list(
                q=search_query,
                part="id",
                maxResults=1,  # We only need the top result
                type="video"
            ).execute()

            # Check if search found anything
            if not search_response['items']:
                print(f"  [{i}/{len(songs)}] Not found on YouTube: {song}")
                continue

            # Get the video ID of the best match
            video_id = search_response['items'][0]['id']['videoId']

            # Add this video to the playlist
            self.youtube.playlistItems().insert(
                part="snippet",
                body={
                    "snippet": {
                        "playlistId": playlist_id,
                        "resourceId": {
                            "kind": "youtube#video",
                            "videoId": video_id
                        }
                    }
                }
            ).execute()

            print(f"  [{i}/{len(songs)}] Added: {song}")

            # Rate limiting: YouTube API has quota limits
            # 0.5s delay keeps us well under the limit
            time.sleep(0.5)

        print("All done! Check your YouTube playlists.")


print("YouTubePlaylistCreator class defined!")
print("   Usage: creator = YouTubePlaylistCreator('client_secret.json')")
print("          playlist_id = creator.create_playlist('My Mix')")
print("          creator.add_songs(playlist_id, songs)")



YouTubePlaylistCreator class defined!
   Usage: creator = YouTubePlaylistCreator('client_secret.json')
          playlist_id = creator.create_playlist('My Mix')
          creator.add_songs(playlist_id, songs)


In [10]:
# STEP 5: EXTRACT SONGS FROM YOUR YOUTUBE MIX

# PASTE YOUR YOUTUBE MIX URL HERE:
MIX_URL = "https://www.youtube.com/watch?v=466wtWWinlk&list=RDsuRE1UX4Z8k&index=1"


# Create extractor instance
extractor = YouTubeExtractor()

# Extract up to 10 songs for testing (i will it change to 50 for full mix)
songs = extractor.extract_mix(MIX_URL, max_songs=80)

# Display what we found
print("Extracted songs:")
for i, song in enumerate(songs, 1):
    print(f"   {i}. {song}")

# Save backup to JSON (optional, but recommended)
if songs:
    backup_data = [{
        "title": s.title,
        "artist": s.artist,
        "youtube_id": s.youtube_id,
        "duration_seconds": s.duration
    } for s in songs]

    with open("extracted_songs.json", "w", encoding="utf-8") as f:
        json.dump(backup_data, f, indent=2, ensure_ascii=False)

    print("Backup saved to: extracted_songs.json")



Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


Extracting from: https://www.youtube.com/watch?v=466wtWWinlk&list=RDsuRE1UX4Z...
Extracted 80 songs from the mix!
Extracted songs:
   1. Unknown - Berth
   2. Hazlett - Blame The Moon (Live from Montreal)
   3. Unknown - Sweet Heat Lightning
   4. Gregory Alan Isakov - Mistakes (Official Music Video)
   5. Zackery - Do Your Worst (Official Video)
   6. Gregory Alan Isakov - San Luis (OFFICIAL VIDEO)
   7. Ben Howard - Promise
   8. Hazlett - I'm Not Ready To Go (Official Music Video)
   9. Hollow Coves - These Memories (Official Music Video)
   10. Hayd, Chance Peña - How Long, How Low? (Official Music Video)
   11. Unknown - Powder
   12. Harrison Storm - Moon and Back
   13. Hollow Coves - Blessings (Official Music Video)
   14. Hazlett - blue jean (Official Audio)
   15. Unknown - Gert Taberner "Fallen" (Official Music Video)
   16. Unknown - Slow It Down
   17. City and Colour - Things We Choose To Care About (Official Music Video)
   18. Hazlett - Blame The Moon (Official Lyric Vi

In [11]:
# STEP 6: I WILL CREATE PLAYLIST AND ADD SONGS

# Initialize the API client (this opens a browser for login)
creator = YouTubePlaylistCreator("client_secret.json")

# Create a new playlist
playlist_name = "My Converted Mix"  # Change this to whatever you want!
playlist_id = creator.create_playlist(
    title=playlist_name,
    description="Auto-generated from YouTube mix",
    privacy="private"  # Options: "private", "unlisted", "public"
)

# Add all extracted songs to the playlist
creator.add_songs(playlist_id, songs)

print(f"Success! Your playlist '{playlist_name}' is ready on YouTube.")



Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=108415313561-k2587lq88lf7ch5sqbphuaidqifu0b49.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55807%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube&state=HNK0khFRI9TAhRCvTqRaIgWcvlcznM&code_challenge=TP0W4EVpzwO9dGYddB4rG9aMxZO9M2-AxIkAPTUSCa4&code_challenge_method=S256&access_type=offline
Successfully authenticated with YouTube API!
Created YouTube playlist: 'My Converted Mix'
   Privacy: private
   ID: PLDW0hX-ksmz0CgXaWT1sBgMZUSSnlF9Xb
Adding 80 songs to playlist...
  [1/80] Added: Unknown - Berth
  [2/80] Added: Hazlett - Blame The Moon (Live from Montreal)
  [3/80] Added: Unknown - Sweet Heat Lightning
  [4/80] Added: Gregory Alan Isakov - Mistakes (Official Music Video)
  [5/80] Added: Zackery - Do Your Worst (Official Video)
  [6/80] Added: Gregory Alan Isakov - San Luis (OFFICIAL VIDEO)
  [7/80] Added: Ben Howard - Promise
  [8/

HttpError: <HttpError 403 when requesting https://youtube.googleapis.com/youtube/v3/search?q=City+and+Colour+Things+We+Choose+To+Care+About+%28Official+Music+Video%29&part=id&maxResults=1&type=video&alt=json returned "The request cannot be completed because you have exceeded your <a href="/youtube/v3/getting-started#quota">quota</a>.". Details: "[{'message': 'The request cannot be completed because you have exceeded your <a href="/youtube/v3/getting-started#quota">quota</a>.', 'domain': 'youtube.quota', 'reason': 'quotaExceeded'}]">

In [7]:
# Check what happened
print(f"songs variable exists: {'songs' in globals()}")
if 'songs' in globals():
    print(f"Number of songs: {len(songs)}")
    if songs:
        for i, s in enumerate(songs[:5], 1):
            print(f"  {i}. {s}")
    else:
        print("  (songs list is EMPTY)")
else:
    print("  songs variable NOT FOUND - Cell 6 didn't run properly")

songs variable exists: True
Number of songs: 0
  (songs list is EMPTY)
